# DSN Bootcamp Qualification Hackathon 2026
## Phase 1: Data Audit

The objective of this phase is to understand the structure and quality
of the training and test datasets before performing EDA, feature
engineering, validation, or model training.

### Audit objectives

1. Dataset structure
2. Data types
3. Missing values
4. Cardinality
5. Duplicate observations
6. Train/test consistency
7. Target variable audit


In [ ]:
# Import and load data
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Sum of Train duplicates", train.duplicated().sum())
train

In [ ]:
# Inspect Columns
print("Train Columns")

for i, col in enumerate(train.columns, start=1):
    print(f"{i:2}. {col}")


print("\nTest Columns")

for i, col in enumerate(test.columns, start=1):
    print(f"{i:2}. {col}")

In [ ]:
# Basic Structure
print("Train Info")
train.info()

print("\nTest Info")
test.info()

In [ ]:
# Data Type Audit
dtype_audit = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing": train.isna().sum(),
    "missing_pct": (train.isna().mean() * 100).round(2),
    "unique_values": train.nunique(dropna=True)
})

dtype_audit

In [ ]:
#Missing-value Audit
missing_audit = pd.DataFrame(
    {
    "train_missing": train.isna().sum(),
    "train_missing_pct": (train.isna().mean() * 100).round(2),
    "test_missing": test.isna().sum(),
    "test_missing_pct": (test.isna().mean() * 100).round(2)
    }
)

missing_audit = missing_audit.sort_values(
    "train_missing",
    ascending=False
)

missing_audit

In [ ]:
#Total Missing Values
print("Total missing cells:")

print("Train:", train.isna().sum().sum())
print("Test :", test.isna().sum().sum())

In [ ]:
# Unique Value Audit
cardinality = pd.DataFrame(
    {
    "dtype": train.dtypes.astype(str),
    "n_unique": train.nunique(dropna=True),
    "unique_pct": (train.nunique(dropna=True) / len(train) * 100).round(2)
    }
    )

cardinality = cardinality.sort_values("n_unique", ascending = False)

cardinality

In [ ]:
# Exact duplicates
print("Exact Duplicate Rows:")

print("Train:", train.duplicated().sum())
print("Test :", test.duplicated().sum())

In [ ]:
# Product Store duplicates
pair_columns = ["product_code", "store_code"]

train_pair_duplicates = train.duplicated(subset=pair_columns).sum()

test_pair_duplicates = test.duplicated(
    subset=pair_columns
).sum()

print("Duplicate Product-Store Combinations")

print("Train:", train_pair_duplicates)
print("Test :", test_pair_duplicates)

In [ ]:
# Identify Categorical Columns
categorical_columns = train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

In [ ]:
# Compare Categorical Domains and discover if categories exist in test but not train
for column in categorical_columns:

    train_values = set(train[column].dropna().unique())
    test_values = set(test[column].dropna().unique())

    test_only_values = test_values - train_values

    print(f"\n{column}")
    print("Train unique values:", len(train_values))
    print("Test unique values :", len(test_values))
    print("Test-only values   :", test_only_values)

In [ ]:
# Target Statistics
target = "total_sales"

print("TARGET:", target)
display(train[target].describe().to_frame().T)

In [ ]:
# Target Validity
print("Missing target values :", train[target].isna().sum())
print("Zero target values    :", (train[target] == 0).sum())
print("Negative target values:", (train[target] < 0).sum())
print("Unique target values  :", train[target].nunique())

In [ ]:
# Target Quantiles
target_quantiles = train[target].quantile(
    [ 0.00, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
)

target_quantiles

In [ ]:
# Dataset Contract
target = "total_sales"

# Target exists only in train
assert target in train.columns
assert target not in test.columns

# Train and test predictors must match
train_features = set(train.columns) - {target}
test_features = set(test.columns)

assert train_features == test_features

# IDs should be unique
assert train["id"].is_unique
assert test["id"].is_unique

# Target should not contain missing values
assert train[target].notna().all()

print("Target correctly identified.")
print("Train/test predictor columns match.")
print("Train IDs are unique.")
print("Test IDs are unique.")
print("Training target contains no missing values.")
print("\nDataset Contract Passed.")

In [32]:
# Category integrity audit

category_columns = [
    "fat_content",
    "product_category",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

for col in category_columns:
    print(f"{col}")

    print("\nTrain:")
    display(train[col].value_counts(dropna=False).to_frame("count"))

    print("\nTest:")
    display(test[col].value_counts(dropna=False).to_frame("count"))

fat_content

Train:


,count
fat_content,
Low Fat,4425
Regular,2393



Test:


,count
fat_content,
Low Fat,1092
Regular,613


product_category

Train:


...

,count
store_location_tier,
Tier_3,673
Tier_2,553
Tier_1,479


store_format

Train:


,count
store_format,
Standard Supermarket,4462
Corner Shop,866
Flagship Hypermarket,748
Superstore,742



Test:


,count
store_format,
Standard Supermarket,1115
Corner Shop,217
Flagship Hypermarket,187
Superstore,186


In [34]:
# Check for possible whitespace/case inconsistencies

for col in [
    "fat_content",
    "product_category",
    "store_size",
    "store_location_tier",
    "store_format",
]:
    values = pd.concat(
        [train[col], test[col]],
        ignore_index=True
    ).dropna().astype(str)

    normalized = values.str.strip().str.lower()

    variants = pd.DataFrame({
        "raw": values,
        "normalized": normalized
    }).drop_duplicates()

    grouped = variants.groupby("normalized")["raw"].agg(list)

    inconsistent = grouped[grouped.str.len() > 1]

    print(f"{col}")

    if len(inconsistent) == 0:
        print("No obvious case/whitespace variants found.")
    else:
        display(inconsistent.to_frame("variants"))

fat_content
No obvious case/whitespace variants found.
product_category


,variants
normalized,
baking goods,"[baking goods, BAKING GOODS, Baking Goods]"
breads,"[Breads, BREADS, breads]"
breakfast,"[Breakfast, breakfast, BREAKFAST]"
canned,"[Canned, CANNED, canned]"
dairy,"[Dairy, dairy, DAIRY]"
frozen foods,"[Frozen Foods, frozen foods, FROZEN FOODS]"
fruits and vegetables,"[Fruits and Vegetables, fruits and vegetables,..."
hard drinks,"[Hard Drinks, hard drinks, HARD DRINKS]"
health and hygiene,"[HEALTH AND HYGIENE, Health and Hygiene, healt..."


store_size
No obvious case/whitespace variants found.
store_location_tier
No obvious case/whitespace variants found.
store_format
No obvious case/whitespace variants found.


In [35]:
# Product code structure audit

print("Number of unique product codes:")
print(train["product_code"].nunique())

print("\nSample product codes:")
display(train["product_code"].head(20).to_frame())

print("\nProduct code length distribution:")
display(
    train["product_code"]
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
    .to_frame("count")
)

print("\nProduct code prefix distribution:")
display(
    train["product_code"]
    .astype(str)
    .str[:2]
    .value_counts()
    .head(30)
    .to_frame("count")
)

Number of unique product codes:
1555

Sample product codes:


,product_code
0,PRD-PRFP9S
1,PRD-PXXK71
2,PRD-V5MOIJ
3,PRD-UN5Z3J
4,PRD-6RDQYB
5,PRD-E6VA05
6,PRD-MHLD93
7,PRD-EF6Y39
8,PRD-IAZMTU
9,PRD-DH2TA3



Product code length distribution:


,count
product_code,
10,6818



Product code prefix distribution:


,count
product_code,
PR,6818


In [36]:
# Check whether product codes have a stable relationship with product categories

product_category_mapping = (
    train.groupby("product_code")["product_category"]
    .nunique()
)

print("Product codes associated with more than one raw category:")
print((product_category_mapping > 1).sum())

print("\nMaximum number of raw categories associated with one product code:")
print(product_category_mapping.max())

Product codes associated with more than one raw category:
1392

Maximum number of raw categories associated with one product code:
3


In [38]:
# Product-level attribute consistency audit

product_attributes = [
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_category",
    "product_price",
]

for col in product_attributes:
    consistency = train.groupby("product_code")[col].nunique(dropna=False)

    print(col)
    print("Products with exactly one unique value:", (consistency == 1).sum())
    print("Products with more than one unique value:", (consistency > 1).sum())
    print("Maximum unique values for one product:", consistency.max())

product_weight_kg
Products with exactly one unique value: 47
Products with more than one unique value: 1508
Maximum unique values for one product: 9
fat_content
Products with exactly one unique value: 1555
Products with more than one unique value: 0
Maximum unique values for one product: 1
shelf_visibility
Products with exactly one unique value: 47
Products with more than one unique value: 1508
Maximum unique values for one product: 9
product_category
Products with exactly one unique value: 163
Products with more than one unique value: 1392
Maximum unique values for one product: 3
product_price
Products with exactly one unique value: 46
Products with more than one unique value: 1509
Maximum unique values for one product: 9


In [39]:
# Number of stores associated with each product

stores_per_product = train.groupby("product_code")["store_code"].nunique()

print("Distribution of number of stores per product:")
display(stores_per_product.value_counts().sort_index().to_frame("number_of_products"))

print("\nMaximum stores observed for one product:", stores_per_product.max())
print("Products appearing in all 10 stores:", (stores_per_product == 10).sum())

Distribution of number of stores per product:


,number_of_products
store_code,
1,46
2,140
3,251
4,394
5,351
6,240
7,97
8,35
9,1



Maximum stores observed for one product: 9
Products appearing in all 10 stores: 0


# Findings & Inferences

The data audit established the structural properties and quality of the
training and test datasets before any transformation, imputation,
encoding, or feature engineering was applied.

## Key Findings

### 1. Dataset structure

- The training set contains 6,818 observations and 13 columns.
- The test set contains 1,705 observations and 12 columns.
- The predictor columns are consistent between train and test.
- Train and test IDs are unique.
- There are no exact duplicate rows.
- There are no duplicate `product_code + store_code` combinations.
- The target variable, `total_sales`, is present only in the training set
  and contains no missing or invalid values.

### 2. Missing data

Missing values are concentrated in two predictors:

- `product_weight_kg`: approximately 18% missing
- `store_size`: approximately 28% missing

The missingness rates are similar between the training and test sets,
suggesting that missing values are part of the dataset structure rather
than an obvious train/test discrepancy.

The missing values will therefore be investigated further before deciding
how they should be handled.

### 3. Categorical data integrity

`product_category` contains systematic capitalization differences.
For example, the same semantic category appears as:

- `Fruits and Vegetables`
- `fruits and vegetables`
- `FRUITS AND VEGETABLES`

This creates artificial categorical levels. These representations
should be normalized during preprocessing so that semantically identical
categories are treated consistently.

Other major categorical variables, including `fat_content`, `store_code`,
`store_location_tier`, and `store_format`, do not show the same obvious
case/whitespace inconsistency.

### 4. Product-code structure

There are 1,555 unique `product_code` values.

The codes have a fixed length and share the common `PRD-` prefix, while
the remaining characters appear identifier-like. No obvious semantic
meaning can be inferred from the raw character positions during the audit.

Therefore, individual characters should not automatically be treated as
meaningful features. The predictive value of `product_code` itself will
be investigated empirically.

### 5. Product attributes are not uniformly fixed by product code

An important structural observation emerged when checking attributes
within each `product_code`.

Most product codes are associated with multiple observed values of:

- `product_weight_kg`
- `shelf_visibility`
- `product_price`

In contrast, `fat_content` is perfectly consistent within every
`product_code`.

`product_category` also varies for many product codes, but the earlier
category-integrity audit indicates that much of this variation is caused
by capitalization differences.

The variation in weight, visibility, and price suggests that these
variables may capture context beyond immutable product identity,
potentially including product-store or store-specific effects.

This is an inference rather than a confirmed causal interpretation.
The EDA phase will investigate whether these variables systematically
vary with store characteristics and whether product-store interactions
are useful for prediction.

### 6. Product-store coverage

Products do not appear in every store.

The number of stores associated with a product ranges from 1 to 9,
with no product appearing in all 10 stores.

Therefore, the dataset does not form a complete product × store matrix.
This structure will be considered when designing validation strategies
and investigating product-store relationships.

### 7. Target variable

`total_sales` is strictly positive and right-skewed:

- Mean: approximately 2,174.76
- Median: approximately 1,790.89
- Standard deviation: approximately 1,697.89
- Minimum: 32.70
- Maximum: 12,996.82

Because the competition metric is RMSE, large prediction errors in the
upper tail can have a substantial effect on model performance.

A log-transformed target may therefore be worth testing later, but it
will not be assumed to be superior. Any target transformation will be
evaluated using RMSE on the original sales scale.

## Modeling Implications

The audit suggests several questions for the next phases:

1. How strongly do store characteristics relate to sales?
2. How strongly do product characteristics relate to sales?
3. Are price, visibility, and weight systematically associated with
   particular stores or store formats?
4. Can missing `store_size` or `product_weight_kg` values be informed by
   other variables?
5. Does `product_code` provide useful predictive information despite its
   high cardinality?
6. Are product-store interactions important?
7. How should validation account for repeated products across stores?
8. Do nonlinear relationships justify tree-based boosting models?
9. Does transforming the target improve RMSE on the original sales scale?

These questions will guide the exploratory analysis, validation design,
feature engineering, and model experiments that follow.

> **Conclusion:** The dataset is structurally consistent between
> train and test, but contains meaningful missingness, high-cardinality
> product identifiers, categorical inconsistencies, and substantial
> variation in several product attributes. These characteristics warrant
> investigation before modeling rather than being handled with generic
> preprocessing assumptions.